# Vanilla Gap-K% on StarCoder2-3B (Kaggle)
Enable a **GPU accelerator** and enable **Internet** for the initial model/dataset download, unless cached assets are attached as a private Kaggle Dataset. The first download needs time and several GB of disk. This notebook only orchestrates the installed `code_mia` package; it contains no attack or dataset-processing implementation.

In [ ]:
from pathlib import Path
import json, os, subprocess
REPOSITORY_URL = "REPLACE_WITH_YOUR_GITHUB_REPOSITORY_URL"
REPO = Path("/kaggle/working/code-mia-gapk")
if not (REPO / "pyproject.toml").exists():
    if REPOSITORY_URL.startswith("REPLACE_WITH"):
        raise FileNotFoundError(f"Upload the repository to {REPO}, or replace REPOSITORY_URL with a real Git URL.")
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPO)], check=True)
os.chdir(REPO)
print(REPO)

In [ ]:
os.environ.update({
    "HF_HOME": "/kaggle/working/hf-cache",
    "HUGGINGFACE_HUB_CACHE": "/kaggle/working/hf-cache/hub",
    "HF_DATASETS_CACHE": "/kaggle/working/hf-cache/datasets",
    "TOKENIZERS_PARALLELISM": "false",
})

In [ ]:
subprocess.run(["python", "-m", "pip", "install", "-e", "."], check=True)

In [ ]:
subprocess.run(["nvidia-smi"], check=False)
report_run = subprocess.run(["python", "scripts/inspect_environment.py"], check=True, capture_output=True, text=True)
print(report_run.stdout)
environment = json.loads(report_run.stdout)
if not environment["internet_dns_available"]:
    model_hint = environment.get("model_snapshot_hint") or {}
    dataset_hint = environment.get("dataset_snapshot_hint") or {}
    if not (model_hint.get("exists") and dataset_hint.get("exists")):
        raise RuntimeError("Internet/DNS is unavailable and pinned model/dataset snapshots are not both present in the Hugging Face cache. Attach local snapshots and pass --model-path/--dataset-path, or enable Kaggle Internet for the first download.")

In [ ]:
subprocess.run(["pytest", "-q"], check=True)

## Build and audit the frozen 20+20 smoke manifest

In [ ]:
subprocess.run(["code-mia", "build-manifest", "--config", "configs/smoke.yaml"], check=True)
import json
print(json.dumps(json.load(open("outputs/smoke/dataset_audit.json")), indent=2))

In [ ]:
subprocess.run(["code-mia", "run", "--config", "configs/smoke.yaml"], check=True)

In [ ]:
subprocess.run(["code-mia", "validate-run", "--run-dir", "outputs/smoke"], check=True)
import pandas as pd
from IPython.display import Image, display
print(open("outputs/smoke/metrics.md").read())
display(pd.read_csv("outputs/smoke/scores.csv").head())
display(Image("outputs/smoke/roc_curve.png"), Image("outputs/smoke/score_histogram.png"))
scores = pd.read_csv("outputs/smoke/scores.csv")
print("Output:", (REPO / "outputs/smoke").resolve())
print("Peak allocated bytes:", scores.peak_allocated_gpu_bytes.max())
print("Summed sample runtime seconds:", scores.runtime_seconds.sum())

## Optional longer 100+100 debug run
Run this separately when ready. It is a debugging experiment, not a definitive scientific result.

In [ ]:
subprocess.run(["code-mia", "run", "--config", "configs/debug_gapk_starcoder2_3b.yaml"], check=True)

In [ ]:
DEBUG = REPO / "outputs/debug_gapk_starcoder2_3b"
subprocess.run(["code-mia", "validate-run", "--run-dir", str(DEBUG)], check=True)
print((DEBUG / "metrics.md").read_text())
display(pd.read_csv(DEBUG / "scores.csv").head())
display(Image(str(DEBUG / "roc_curve.png")), Image(str(DEBUG / "score_histogram.png")))

In [ ]:
import shutil
archive = shutil.make_archive("/kaggle/working/code-mia-gapk-results", "zip", REPO / "outputs")
print(archive)